# E0006 — offline import smoke (NO GPU)

Purpose: prove that the saved CUDA 12.9 wheelhouse can install and import **with Internet OFF** before any further L4 quota is spent.

Required settings:
- Accelerator: **None**
- Internet: **OFF**
- Add as Input the saved output of the wheelhouse-builder notebook containing `e0006_cu129_wheelhouse/`

This notebook does **not** load the Nemotron checkpoint and does **not** use a GPU.

Expected output: `/kaggle/working/e0006_offline_import_smoke.json`


In [ ]:

from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

OUT = Path("/kaggle/working/e0006_offline_import_smoke.json")
INPUT_ROOT = Path("/kaggle/input")
RUNTIME = Path("/tmp/e0006_runtime")

EXPECTED_VLLM_SHA256 = "bf0d52faa2a51e7a01c6856a7a8a2d1307fd0ff711415d34168a67ffac0fa47b"
EXPECTED_CUBIN_SHA256 = "c79fba990aee2a7c7ef64208bb65900e45fe23c3a223f3dfc21eef225f43cba2"
MIN_FREE_GIB = 16.0

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def tree_bytes(path: Path) -> int:
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            try:
                total += p.stat().st_size
            except OSError:
                pass
    return total

vllm_candidates = sorted(INPUT_ROOT.rglob("vllm-0.27.1+cu129-*.whl"))
cubin_candidates = sorted(INPUT_ROOT.rglob("flashinfer_cubin-0.6.16.post3-*.whl"))

payload = {
    "experiment": "E0006",
    "gate": "D3_OFFLINE_IMPORT_SMOKE_NO_GPU",
    "python": sys.version,
    "input_root": str(INPUT_ROOT),
    "vllm_candidates": [str(p) for p in vllm_candidates],
    "flashinfer_cubin_candidates": [str(p) for p in cubin_candidates],
    "status": "BLOCKED_INPUT",
}

if len(vllm_candidates) != 1 or len(cubin_candidates) != 1:
    OUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(json.dumps(payload, indent=2, sort_keys=True))
    raise SystemExit("Attach exactly one saved wheelhouse output before running this notebook.")

vllm_wheel = vllm_candidates[0]
cubin_wheel = cubin_candidates[0]
wheelhouse = vllm_wheel.parent

payload["wheelhouse"] = str(wheelhouse)
payload["wheelhouse_file_count"] = len(list(wheelhouse.glob("*.whl")))
payload["vllm_sha256"] = sha256(vllm_wheel)
payload["flashinfer_cubin_sha256"] = sha256(cubin_wheel)
payload["vllm_sha256_ok"] = payload["vllm_sha256"] == EXPECTED_VLLM_SHA256
payload["flashinfer_cubin_sha256_ok"] = payload["flashinfer_cubin_sha256"] == EXPECTED_CUBIN_SHA256

# Verify that the exact architecture name appears in the frozen vLLM wheel code.
arch_hits = []
with zipfile.ZipFile(vllm_wheel) as zf:
    for name in zf.namelist():
        if not name.endswith(".py"):
            continue
        try:
            data = zf.read(name)
        except Exception:
            continue
        if b"NemotronHForCausalLM" in data:
            arch_hits.append(name)
payload["nemotron_architecture_hits_in_vllm_wheel"] = arch_hits
payload["nemotron_architecture_registered_in_wheel"] = bool(arch_hits)

free_tmp = shutil.disk_usage("/tmp").free
free_work = shutil.disk_usage("/kaggle/working").free
payload["free_tmp_gib_before"] = round(free_tmp / 1024**3, 3)
payload["free_working_gib_before"] = round(free_work / 1024**3, 3)

if free_tmp / 1024**3 < MIN_FREE_GIB:
    payload["status"] = "BLOCKED_DISK"
    payload["decision"] = f"Need at least {MIN_FREE_GIB:.1f} GiB free in /tmp before isolated install."
    OUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(json.dumps(payload, indent=2, sort_keys=True))
    raise SystemExit(payload["decision"])

if RUNTIME.exists():
    shutil.rmtree(RUNTIME)
RUNTIME.mkdir(parents=True)

install_cmd = [
    sys.executable, "-m", "pip", "install",
    "--no-index",
    "--find-links", str(wheelhouse),
    "--target", str(RUNTIME),
    "--ignore-installed",
    "--no-cache-dir",
    "--no-compile",
    str(vllm_wheel),
    "flashinfer-cubin==0.6.16.post3",
]
payload["install_command"] = install_cmd

t0 = time.time()
cp = subprocess.run(install_cmd, capture_output=True, text=True, timeout=2400)
payload["install_seconds"] = round(time.time() - t0, 3)
payload["install_returncode"] = cp.returncode
payload["install_stdout_tail"] = cp.stdout[-12000:]
payload["install_stderr_tail"] = cp.stderr[-12000:]
payload["runtime_installed_bytes"] = tree_bytes(RUNTIME)
payload["runtime_installed_gib"] = round(payload["runtime_installed_bytes"] / 1024**3, 3)
payload["free_tmp_gib_after_install"] = round(shutil.disk_usage("/tmp").free / 1024**3, 3)

if cp.returncode != 0:
    payload["status"] = "BLOCKED_INSTALL"
    OUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(json.dumps(payload, indent=2, sort_keys=True))
    raise SystemExit("Offline isolated install failed.")

probe_code = r"""
import importlib.metadata as md
import json

result = {"versions": {}, "imports": {}}
for dist in [
    "vllm", "torch", "transformers", "triton",
    "flashinfer-python", "flashinfer-cubin", "safetensors",
]:
    try:
        result["versions"][dist] = md.version(dist)
    except Exception as e:
        result["versions"][dist] = None

try:
    import torch
    result["imports"]["torch"] = True
    result["torch_cuda_build"] = torch.version.cuda
    result["torch_cuda_available"] = bool(torch.cuda.is_available())
except Exception as e:
    result["imports"]["torch"] = False
    result["torch_error"] = f"{type(e).__name__}: {e}"

try:
    import transformers
    result["imports"]["transformers"] = True
except Exception as e:
    result["imports"]["transformers"] = False
    result["transformers_error"] = f"{type(e).__name__}: {e}"

try:
    import flashinfer
    result["imports"]["flashinfer"] = True
except Exception as e:
    result["imports"]["flashinfer"] = False
    result["flashinfer_error"] = f"{type(e).__name__}: {e}"

try:
    import vllm
    result["imports"]["vllm"] = True
except Exception as e:
    result["imports"]["vllm"] = False
    result["vllm_error"] = f"{type(e).__name__}: {e}"

try:
    from vllm.model_executor.models import ModelRegistry
    result["imports"]["vllm_model_registry"] = True
    result["model_registry_type"] = type(ModelRegistry).__name__
except Exception as e:
    result["imports"]["vllm_model_registry"] = False
    result["model_registry_error"] = f"{type(e).__name__}: {e}"

print(json.dumps(result, sort_keys=True))
"""

env = os.environ.copy()
env["PYTHONPATH"] = str(RUNTIME) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
env["HF_HUB_OFFLINE"] = "1"
env["TRANSFORMERS_OFFLINE"] = "1"
env["VLLM_NO_USAGE_STATS"] = "1"

t1 = time.time()
probe = subprocess.run(
    [sys.executable, "-c", probe_code],
    capture_output=True,
    text=True,
    env=env,
    timeout=300,
)
payload["import_probe_seconds"] = round(time.time() - t1, 3)
payload["import_probe_returncode"] = probe.returncode
payload["import_probe_stdout"] = probe.stdout[-16000:]
payload["import_probe_stderr"] = probe.stderr[-16000:]

probe_payload = None
if probe.stdout.strip():
    try:
        probe_payload = json.loads(probe.stdout.strip().splitlines()[-1])
    except Exception:
        probe_payload = None
payload["import_probe"] = probe_payload

versions = (probe_payload or {}).get("versions", {})
imports = (probe_payload or {}).get("imports", {})
critical_versions_ok = (
    versions.get("vllm") == "0.27.1+cu129"
    and versions.get("torch") == "2.13.0+cu129"
    and versions.get("flashinfer-python") == "0.6.16.post3"
    and versions.get("flashinfer-cubin") == "0.6.16.post3"
)
critical_imports_ok = all(
    imports.get(k) is True
    for k in ["torch", "transformers", "flashinfer", "vllm", "vllm_model_registry"]
)

ready = all([
    payload["vllm_sha256_ok"],
    payload["flashinfer_cubin_sha256_ok"],
    payload["nemotron_architecture_registered_in_wheel"],
    cp.returncode == 0,
    probe.returncode == 0,
    critical_versions_ok,
    critical_imports_ok,
])

payload["critical_versions_ok"] = critical_versions_ok
payload["critical_imports_ok"] = critical_imports_ok
payload["status"] = "OFFLINE_IMPORT_READY" if ready else "BLOCKED_IMPORT"
payload["next_rule"] = (
    "If OFFLINE_IMPORT_READY, review runtime installed size and import details before authorizing any L4 Gate B."
)

OUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps({
    "status": payload["status"],
    "wheelhouse": payload["wheelhouse"],
    "wheelhouse_file_count": payload["wheelhouse_file_count"],
    "runtime_installed_gib": payload["runtime_installed_gib"],
    "free_tmp_gib_after_install": payload["free_tmp_gib_after_install"],
    "critical_versions_ok": payload["critical_versions_ok"],
    "critical_imports_ok": payload["critical_imports_ok"],
    "nemotron_architecture_hits_in_vllm_wheel": payload["nemotron_architecture_hits_in_vllm_wheel"],
    "import_probe": payload["import_probe"],
}, indent=2, sort_keys=True))
print(f"\nWROTE: {OUT}")
